# Arabic Text Error Detection, Correction, and NLP Pipeline
This notebook implements an end-to-end NLP pipeline for Arabic text using open-source libraries (Transformers, Camel-Tools, PyArabic, Qalsadi, NLTK).

**Pipeline Steps:**
1. Installations & Setup
2. Language Verification
3. Contextual Grammar & Spelling Correction (AraT5 Seq2Seq)
4. Visual Error Detection (Color-coded differences)
5. Text Normalization
6. Stemming & Lemmatization

In [ ]:
# Install the required open-source libraries
!pip install transformers torch
!pip install langdetect
!pip install pyarabic camel-tools
!pip install qalsadi nltk

# Core Arabic NLP toolkit (morphology, disambiguation, lemmatization, reinflection)
# + spelling (symspellpy), stemming (tashaphyne, pulls in pyarabic), language ID (langdetect)
!pip install -q camel-tools symspellpy tashaphyne langdetect pandas

import sys
print(f"Python: {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("camel-tools needs Python 3.11+. In Colab: Runtime > Change runtime type.")

!camel_data -i morphology-db-msa-r13
!camel_data -i disambig-mle-calima-msa-r13

!wget -q -O ar_50k.txt "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2016/ar/ar_50k.txt"
!wc -l ar_50k.txt

# Download NLTK tokenization data
import nltk
nltk.download('punkt')

Python: 3.12.13
The following packages will be installed: 'morphology-db-msa-r13'
Extracting package 'morphology-db-msa-r13': 100% 40.5M/40.5M [00:00<00:00, 365MB/s]
The following packages will be installed: 'disambig-mle-calima-msa-r13'
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:00<00:00, 234MB/s]
50000 ar_50k.txt


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import re
import html
import pandas as pd
from IPython.display import display, HTML

from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.morphology.generator import Generator
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar,
)
import pyarabic.araby as araby
from tashaphyne.stemming import ArabicLightStemmer
from symspellpy import SymSpell, Verbosity
from langdetect import detect_langs, DetectorFactory, LangDetectException
DetectorFactory.seed = 0  # deterministic langdetect results

print("Loading morphological analyzer ...")
analyzer = Analyzer(MorphologyDB.builtin_db())

print("Loading morphological generator ...")
generator = Generator(MorphologyDB.builtin_db(flags='g'))

print("Loading MLE disambiguator (POS / gender / number / lemma) ...")
mle_disambiguator = MLEDisambiguator.pretrained()

print("Loading spelling frequency dictionary ...")
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary('ar_50k.txt', term_index=0, count_index=1, separator=' ', encoding='utf-8')

light_stemmer = ArabicLightStemmer()

print("Ready.")

Loading morphological analyzer ...
Loading morphological generator ...
Loading MLE disambiguator (POS / gender / number / lemma) ...
Loading spelling frequency dictionary ...
Ready.


## Step 1: Language Verification
We use `langdetect` to analyze the text and ensure the user has inputted Arabic (`ar`) before proceeding.

In [ ]:
from langdetect import detect, LangDetectException

def verify_arabic(text):
    try:
        lang = detect(text)
        if lang == 'ar':
            print("✅ Language verified: Arabic")
            return True
        else:
            print(f"❌ Text is not Arabic. Detected language: {lang}")
            return False
    except LangDetectException:
        print("❌ Could not detect language. The text might be too short or invalid.")
        return False

# Feel free to change this test sentence to test different errors!
# This sentence contains spelling/grammar mistakes: "سعيداً" should be "سعيدٌ", "الي المدرسه" should be "إلى المدرسة", etc.
user_text = "اكلت الغنت النفاحة وهو سغيدة"

is_arabic = verify_arabic(user_text)

✅ Language verified: Arabic


In [ ]:
ARABIC_LETTERS_RE = re.compile(r'[\u0621-\u063A\u0641-\u064A\u066E\u066F\u0671-\u06D3\u06D5]')
LATIN_RE = re.compile(r'[A-Za-z]')

def is_arabic_token(tok):
    return bool(ARABIC_LETTERS_RE.search(tok)) and not LATIN_RE.search(tok) and not tok.isdigit()

def tokenize_arabic(text):
    return simple_word_tokenize(text)

tokens = tokenize_arabic(user_text)
print(tokens)
print(f"\n{len(tokens)} tokens, {sum(is_arabic_token(t) for t in tokens)} of which are Arabic words")

['اكلت', 'الغنت', 'النفاحة', 'وهو', 'سغيدة']

5 tokens, 5 of which are Arabic words


In [ ]:
def detect_spelling_errors(tokens):
    # Returns {token: [positions]} for tokens with zero morphological analyses.
    errors = {}
    for i, tok in enumerate(tokens):
        if not is_arabic_token(tok):
            continue
        if len(analyzer.analyze(tok)) == 0:
            errors.setdefault(tok, []).append(i)
    return errors

spelling_errors = detect_spelling_errors(tokens)

print(f"{len(spelling_errors)} candidate spelling error(s) found:")
for word, positions in spelling_errors.items():
    print(f"  - '{word}'  at position(s) {positions}")

3 candidate spelling error(s) found:
  - 'الغنت'  at position(s) [1]
  - 'النفاحة'  at position(s) [2]
  - 'سغيدة'  at position(s) [4]


In [ ]:
def correct_spelling(tokens, spelling_errors):
    corrections = {}
    for word in spelling_errors:
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        valid = [s.term for s in suggestions if analyzer.analyze(s.term)]
        if valid:
            corrections[word] = valid[0]
        elif suggestions:
            corrections[word] = suggestions[0].term  # no analyzer-valid candidate; best-effort fallback
        else:
            corrections[word] = None  # nothing found; leave the original word as-is

    corrected_tokens = [
        corrections[tok] if (tok in corrections and corrections[tok]) else tok
        for tok in tokens
    ]
    return corrected_tokens, corrections

spelling_corrected_tokens, spelling_corrections = correct_spelling(tokens, spelling_errors)

for original, fixed in spelling_corrections.items():
    print(f"  '{original}'  ->  '{fixed}'")
print("\nText after spelling correction:")
spell_corrected_text = ' '.join(spelling_corrected_tokens)
print(spell_corrected_text)


  'الغنت'  ->  'البنت'
  'النفاحة'  ->  'التفاحة'
  'سغيدة'  ->  'سعيدة'

Text after spelling correction:
اكلت البنت التفاحة وهو سعيدة


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Note: You must log into Hugging Face using `huggingface-cli login` first
# and accept the Gemma 3 license on their website.
model_name = "alnnahwi/gemma-3-1b-arabic-gec-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Wrap the text in the prompt format the model was trained on
messages = [{"role": "user", "content": spell_corrected_text}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)

corrected_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Corrected:", corrected_text)

config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Corrected: model
أكلت البنت التفاحة وهو سعيد.


## Step 2: Contextual Grammar & Spelling Correction
We use a transformer model from Hugging Face based on **AraT5** (`SuperSl6/Arabic-Text-Correction`), which is specifically fine-tuned for Arabic Sequence-to-Sequence grammar and spelling correction (GEC). It understands the context of the whole sentence.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

user_text = "اكل البنت التفاحة وهو سعيدة"

if is_arabic:
    print("⏳ Loading AraT5 Transformer model (this may take a minute)...")

    # We revert to your original working model
    model_name = "SuperSl6/Arabic-Text-Correction"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def correct_arabic_text(text):
        # Tokenize the input text
        inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)

        # Generate the corrected sequence with STRICT anti-hallucination rules
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=4,                   # Lower beam size to prevent over-complicating the search
            repetition_penalty=2.5,        # Heavily penalizes repeating the same word
            no_repeat_ngram_size=2,        # Physically prevents the model from repeating a 2-word phrase
            early_stopping=True,
            do_sample=False                # Ensures deterministic output (no random guessing)
        )

        # Decode the output back to text
        corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Clean up any extra spaces
        corrected_text = corrected_text.strip()
        return corrected_text

    corrected_text = correct_arabic_text(user_text)

    print("\n📝 Original Text :", user_text)
    print("✨ Corrected Text:", corrected_text)

⏳ Loading AraT5 Transformer model (this may take a minute)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



📝 Original Text : اكل البنت التفاحة وهو سعيدة
✨ Corrected Text: التفاح وهو سعيدة وأكل البنت التفاحة وهو سعيدة وأكل البنت التفيحة وهي سعية وهو ساعدة أكل البنتين التفافاحة وهو سعيدة وهو سعيدة اكل البنة التافة وهو سعدية وهو سعيدة اكل البنت التفاحة ، وهو سيسعىدة واكل الببت التفاية وهو سعادنة وهو سجيدة .


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

user_text = "اكل البنت التفاحة وهو سعيدة"
is_arabic = True # Assuming verified

if is_arabic:
    print("⏳ Loading UBC-NLP AraT5 Transformer...")

    # Using a broader Seq2Seq model that handles contextual grammar better
    model_name = "UBC-NLP/AraT5-base-finetuned-arabic-gec"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def correct_arabic_text(text):
        inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)

        # We apply the STRICT anti-hallucination rules here to prevent looping
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=5,
            repetition_penalty=2.0,        # Prevents repeating words like "وهو وهو"
            no_repeat_ngram_size=2,        # Blocks 2-word phrase loops
            early_stopping=True,
            do_sample=False                # Deterministic output
        )

        corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return corrected_text.strip()

    print("🛠️ Processing text...")
    corrected_text = correct_arabic_text(user_text)

    print("\n📝 Original Text :", user_text)
    print("✨ Corrected Text:", corrected_text)

⏳ Loading UBC-NLP AraT5 Transformer...


OSError: UBC-NLP/AraT5-base-finetuned-arabic-gec is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo with `use_auth_token` or log in with `huggingface-cli login` and pass `use_auth_token=True`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Note: You must log into Hugging Face using `huggingface-cli login` first
# and accept the Gemma 3 license on their website.
model_name = "alnnahwi/gemma-3-1b-arabic-gec-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Wrap the text in the prompt format the model was trained on
messages = [{"role": "user", "content": "البنت اكل البفاحة وهو سعيدة"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)

corrected_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Corrected:", corrected_text)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Corrected: model
أكلت البنت البفاحة وهو سعيد


In [ ]:
# 1. Install camel-tools and download the required morphology databases
!pip install camel-tools torch
!camel_data -i morphology-db-msa-r13
!camel_data -i disambig-mle-msa

# 2. Clone the CAMeL Lab Arabic GEC repository
!git clone https://github.com/CAMeL-Lab/arabic-gec.git

# 3. Install the official, modern transformers first to safely grab ALL working dependencies (including tokenizers)
!pip install transformers

# 4. Uninstall ONLY the official transformers (leaving the working tokenizers installed)
!pip uninstall -y transformers

# 5. Install CAMeL Lab's custom fork WITHOUT triggering dependency checks
!pip install ./arabic-gec/transformers --no-deps

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
No new packages will be installed.
Error: Invalid package name 'disambig-mle-msa'fatal: destination path 'arabic-gec' already exists and is not an empty directory.
Found existing installation: transformers 5.14.1
Uninstalling transformers-5.14.1:
  Successfully uninstalled transformers-5.14.1
Processing ./arabic-gec/transformers
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.22.2-py3-none-any.whl size=5053789 sha256=d220a9223ac7c86dd8b6b6e0cb71788351318bb802381a1cd130761e3e21db23
  Stored in directory: /tmp/pip-ephem-wheel-cache-8dd2zwi9/wheels/be/ae/a2/f4619880431339ef85a2c755bbef2be13a08dc9b0a79ed2ade
Successfully built transformers


In [ ]:
!pip install "huggingface_hub<0.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.22.2 requires tokenizers!=0.11.3,<0.13,>=0.11.1, but you have tokenizers 0.22.2 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.22.2 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.22.2 which is incompatible.
sentence-transformers 5.6.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.22.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.2

In [ ]:
# This overwrites the dependency checking script with an empty file so it skips the version crash
!echo "# bypassed" > /usr/local/lib/python3.12/dist-packages/transformers/dependency_versions_check.py

In [ ]:
# This overwrites the file with a dummy function so the import succeeds but the check is skipped
!echo "def dep_version_check(*args, **kwargs): pass" > /usr/local/lib/python3.12/dist-packages/transformers/dependency_versions_check.py

In [ ]:
!camel_data -i disambig-bert-unfactored-msa

The following packages will be installed: 'disambig-bert-unfactored-msa'
Extracting package 'disambig-bert-unfactored-msa': 100% 445M/445M [00:07<00:00, 61.2MB/s]


In [ ]:
!camel_data -i disambig-mle-calima-msa-r13

The following packages will be installed: 'disambig-mle-calima-msa-r13'
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:01<00:00, 82.4MB/s]


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForTokenClassification, MBartForConditionalGeneration
from camel_tools.disambig.mle import MLEDisambiguator  # <-- Switched to MLE
from camel_tools.utils.dediac import dediac_ar
import warnings
warnings.filterwarnings('ignore') # Hides unnecessary tensor warnings

# Make sure this is set to True (assuming you ran the language check earlier)
is_arabic = True
user_text = "اكل البنت التفاحة وهو سعيدة"

if is_arabic:
    print("⏳ Loading CAMeL Lab Models (This will take a minute on the first run)...")

    # 1. Load Morphological Disambiguator (Using the stable MLE version)
    mle_disambig = MLEDisambiguator.pretrained()

    # 2. Load Error Detection Model (GED)
    ged_tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/camelbert-msa-qalb14-ged-13')
    ged_model = BertForTokenClassification.from_pretrained('CAMeL-Lab/camelbert-msa-qalb14-ged-13')

    # 3. Load Error Correction Model (GEC)
    gec_tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/arabart-qalb14-gec-ged-13')
    gec_model = MBartForConditionalGeneration.from_pretrained('CAMeL-Lab/arabart-qalb14-gec-ged-13')

    def correct_arabic_camel_pipeline(text):
        # --- STAGE 1: Morphological Processing ---
        # The model requires the text to be split into root morphologies
        text_disambig = mle_disambig.disambiguate(text.split())

        morph_pp_text = []
        for w_disambig in text_disambig:
            # If analyses exist, grab the diacritics, otherwise keep the word as is
            if len(w_disambig.analyses) > 0:
                morph_pp_text.append(dediac_ar(w_disambig.analyses[0].analysis['diac']))
            else:
                morph_pp_text.append(w_disambig.word)

        morph_pp_text = ' '.join(morph_pp_text)

        # --- STAGE 2: Grammatical Error Detection (GED) Tagging ---
        inputs = ged_tokenizer([morph_pp_text], return_tensors='pt')
        logits = ged_model(**inputs).logits

        # Extract the predictions, stripping the [CLS] and [SEP] tokens (the [1:-1] slice)
        preds = F.softmax(logits, dim=-1).squeeze()[1:-1]

        # Handle single-word edge cases where squeeze() makes it 1D
        if preds.dim() == 1:
            preds = preds.unsqueeze(0)

        pred_ged_labels = [ged_model.config.id2label[p.item()] for p in torch.argmax(preds, -1)]

        # --- STAGE 3: Map GED Tags to GEC Tokens ---
        ged_label2ids = gec_model.config.ged_label2id
        tokens, ged_labels = [], []

        for word, label in zip(morph_pp_text.split(), pred_ged_labels):
            word_tokens = gec_tokenizer.tokenize(word)
            if len(word_tokens) > 0:
                tokens.extend(word_tokens)
                ged_labels.extend([label for _ in range(len(word_tokens))])

        # Prepare formatting specifically tailored to the AraBART architecture
        input_ids = gec_tokenizer.convert_tokens_to_ids(tokens)
        input_ids = [gec_tokenizer.bos_token_id] + input_ids + [gec_tokenizer.eos_token_id]

        # Fetch label IDs (Defaulting to 0 for Unchanged/Safe tokens)
        label_ids = [ged_label2ids.get(label, 0) for label in ged_labels]
        uc_id = ged_label2ids.get('UC', 0)
        label_ids = [uc_id] + label_ids + [uc_id]

        attention_mask = [1 for _ in range(len(input_ids))]

        # Notice the custom 'ged_tags' parameter - standard Hugging Face fails here!
        gen_kwargs = {
            'num_beams': 5,
            'max_length': 100,
            'num_return_sequences': 1,
            'no_repeat_ngram_size': 0,
            'early_stopping': False,
            'ged_tags': torch.tensor([label_ids]),
            'attention_mask': torch.tensor([attention_mask])
        }

        # --- STAGE 4: Final Generation ---
        generated = gec_model.generate(torch.tensor([input_ids]), **gen_kwargs)
        generated_text = gec_tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

        # Detokenize: CAMeL outputs morphological boundaries using '+', so we clean them up
        cleaned_text = generated_text.replace(" +", "").replace("+ ", "").replace("+", "")

        return cleaned_text.strip()

    print("🛠️ Processing text through pipeline...")
    corrected_text = correct_arabic_camel_pipeline(user_text)

    print("\n📝 Original Text :", user_text)
    print("✨ Corrected Text:", corrected_text)

⏳ Loading CAMeL Lab Models (This will take a minute on the first run)...


tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

🛠️ Processing text through pipeline...

📝 Original Text : اكل البنت التفاحة وهو سعيدة
✨ Corrected Text: أكل البنت التفاحة وهو سعيدة.


## Step 3: Color-Code Errors and Corrections
By comparing the original text with the transformer's output using `difflib`, we can visually detect the exact errors.
*   <span style="color:red">**Red (Strikethrough)**</span> = Detected spelling/grammatical error.
*   <span style="color:green">**Green (Highlighted)**</span> = The system's correction.

In [ ]:
import difflib
from IPython.display import display, HTML

def highlight_differences(original, corrected):
    # Split text into words to compare them one by one
    seq = difflib.SequenceMatcher(None, original.split(), corrected.split())

    # Use HTML/CSS to render a Right-to-Left Arabic display in Colab
    html_output = "<div style='font-size:20px; direction:rtl; text-align:right; font-family:Tahoma, Arial, sans-serif; line-height:2; padding:10px; border:1px solid #ccc; border-radius:5px;'>"

    for opcode, a0, a1, b0, b1 in seq.get_opcodes():
        orig_chunk = ' '.join(original.split()[a0:a1])
        corr_chunk = ' '.join(corrected.split()[b0:b1])

        if opcode == 'equal':
            html_output += f"<span style='color:#333;'> {orig_chunk} </span>"
        elif opcode == 'insert':
            # Word was added (Grammar fix)
            html_output += f"<span style='color:#155724; font-weight:bold; background-color:#d4edda; padding:0 4px; border-radius:3px;'> {corr_chunk} </span>"
        elif opcode == 'delete':
            # Word was removed (Grammar fix)
            html_output += f"<span style='color:#721c24; text-decoration:line-through; background-color:#f8d7da; padding:0 4px; border-radius:3px;'> {orig_chunk} </span>"
        elif opcode == 'replace':
            # Word was replaced (Spelling/Grammar fix)
            html_output += f"<span style='color:#721c24; text-decoration:line-through; background-color:#f8d7da; padding:0 4px; border-radius:3px; margin-left:4px;'> {orig_chunk} </span>"
            html_output += f"<span style='color:#155724; font-weight:bold; background-color:#d4edda; padding:0 4px; border-radius:3px;'> {corr_chunk} </span>"

    html_output += "</div>"
    display(HTML(html_output))

if is_arabic:
    print("🔍 Visualizing detected errors and corrections:")
    highlight_differences(user_text, corrected_text)

🔍 Visualizing detected errors and corrections:


## Step 4: Text Normalization
We use `PyArabic` and `Camel-Tools` to clean the corrected text. This standardizes characters (e.g., merging various shapes of Alef "أ، إ، آ" to a bare Alef "ا") and removes unnecessary diacritics (Tashkeel) or elongations (Tatweel).

In [ ]:
from camel_tools.utils.normalize import normalize_alef_ar, normalize_teh_marbuta_ar
import pyarabic.araby as araby

def normalize_arabic_text(text):
    # 1. Strip diacritics (Tashkeel)
    text = araby.strip_tashkeel(text)
    # 2. Strip elongations (Tatweel / Kashida)
    text = araby.strip_tatweel(text)
    # 3. Standardize all Alef variations (أ, إ, آ) to a plain Alef (ا)
    text = normalize_alef_ar(text)
    # 4. (Optional) Standardize Teh Marbuta to Heh - Uncomment if needed for strict root search
    # text = normalize_teh_marbuta_ar(text)

    return text

if is_arabic:
    normalized_text = normalize_arabic_text(corrected_text)
    print("🔹 Corrected Text  :", corrected_text)
    print("🔹 Normalized Text :", normalized_text)

🔹 Corrected Text  : التفاحة وهو سعيدا جددا وذهب إلى المدرسة وهو سعيدا جدا وذهب إلى المدرسة وهو سعيدا جدا وذهب إلى المدرسة وهو سعيدا جدا وذهب إلى المدرسة وهو سعيدا جدا وذهب إلى المدرسة وهو سعيدا جدا وذهب إلى المدرسة وهو سعيدا جدا وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة وذهب إلى المدرسة
🔹 Normalized Text : التفاحة وهو سعيدا جددا وذهب الى المدرسة وهو سعيدا جدا وذهب الى المدرسة وهو سعيدا جدا وذهب الى المدرسة وهو سعيدا جدا وذهب الى المدرسة وهو سعيدا جدا وذهب الى المدرسة وهو سعيدا جدا وذهب الى المدرسة وهو سعيدا جدا وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة وذهب الى المدرسة


## Step 5: Stemming and Lemmatization
To analyze the morphology of each normalized word, we extract:
*   **Stem**: Uses NLTK's `ISRIStemmer` (strips prefixes and suffixes algebraically).
*   **Lemma**: Uses `Qalsadi`, a robust Arabic morphological analyzer backed by a deep Arabic dictionary, which returns the true dictionary root (e.g., "المدرسة" -> "مدرسة").

In [ ]:
import qalsadi.lemmatizer
from nltk.stem.isri import ISRIStemmer

if is_arabic:
    # Initialize the NLTK Arabic Stemmer
    stemmer = ISRIStemmer()

    # Initialize the Qalsadi Lemmatizer
    lemmer = qalsadi.lemmatizer.Lemmatizer()

    # Tokenize the normalized text into individual words
    words = nltk.word_tokenize(normalized_text)

    print(f"{'Original Word':<18} | {'Stem (ISRI)':<18} | {'Lemma (Qalsadi)':<18}")
    print("-" * 65)

    for word in words:
        stem = stemmer.stem(word)
        lemma = lemmer.lemmatize(word)

        # Print tabular format.
        # Note: RTL Arabic text may look slightly misaligned in a standard LTR console, but the data is accurate.
        print(f"{word:<18} | {stem:<18} | {lemma:<18}")

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
